In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIO\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfter\BoSSSpad.dll"
#r "D:\Users\miao\Documents\JupyterNotebook\BoSSSpadMiao1\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
//ExecutionQueues

In [3]:
//GetDefaultQueue()

In [7]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-FF-Taylor-Couette-Flow");

In [9]:
BoSSSshell.WorkflowMgm.Init("EE-FF-Taylor-Couette-Flow");

Project name is set to 'EE-FF-Taylor-Couette-Flow'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\EE-FF-Taylor-Couette-Flow'.


In [11]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database 'D:\Users\miao\Documents\Database\EE-FF-Taylor-Couette-Flow'.


## Create grid

In [14]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid(int h) { 
        double xLeft = -2;
        double xRight = 2;
        double yTop = 2;
        double yBottom = -2;
        int kelem = Convert.ToInt32(Math.Pow(2, h));;
        
        double[] CutOut1Point1 = new double[2] {  0.5, -0.5 }; 
        double[] CutOut1Point2 = new double[2] {  -0.5, 0.5 };
        
        var CutOut1 = new BoSSS.Platform.Utils.Geom.BoundingBox(2);
        CutOut1.AddPoint(CutOut1Point1);
        CutOut1.AddPoint(CutOut1Point2);

        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, CutOuts: CutOut1);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall_upper");
        //grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
        grd.EdgeTagNames.Add(3, "wall_left");
        grd.EdgeTagNames.Add(4, "wall_right");
        grd.EdgeTagNames.Add(5, "wall_inlet");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 5;

            if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
                et = 4;

            return et;
        });

        return grd;
     }
 
 }

In [16]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocityX(double[] X) {");
           stw.WriteLine("    double A = 0.480186583885618;");
           stw.WriteLine("    double B = 0.079253664457529;");
           stw.WriteLine("    double v_x = -(A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[1] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return v_x;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double InletVelocityY(double[] X) {");
           stw.WriteLine("    double A = 0.480186583885618;");
           stw.WriteLine("    double B = 0.079253664457529;");
           stw.WriteLine("    double v_y = (A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[0] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return v_y;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double PreDisplacementX(double[] X, double t) {");
           stw.WriteLine("    double A = -0.341719453554960;");
           stw.WriteLine("    double B = 1.585073289150575;");
           stw.WriteLine("    double dis_x = -(A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[1] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return dis_x;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double PreDisplacementY(double[] X, double t) {");
           stw.WriteLine("    double A = -0.341719453554960;");
           stw.WriteLine("    double B = 1.585073289150575;");
           stw.WriteLine("    double dis_y = (A * Math.Sqrt(X[0] * X[0] + X[1] * X[1]) + B / Math.Sqrt(X[0] * X[0] + X[1] * X[1])) * X[0] / Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           stw.WriteLine("    return dis_y;");
           stw.WriteLine("  }");
            
           stw.WriteLine("  static public double Pressure(double[] X) {");
           //stw.WriteLine("    double A = 1.047120418848;");
           //stw.WriteLine("    double B = -0.094240837696;");
           //stw.WriteLine("    double rho = 1.0;");
           //stw.WriteLine("    double p0 = 1;");
           //stw.WriteLine("    double r = Math.Sqrt(X[0] * X[0] + X[1] * X[1]);");
           //stw.WriteLine("    double p = p0 + rho * (0.5 * A * A * r * r + 2 * A * B * Math.Log(r) - 0.5 * (B * B) / (r * r));");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return -1;");
           //stw.WriteLine("    return -(X[1] * X[1] + X[0] * X[0] - 0.36);");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return (X[1] * X[1] + X[0] * X[0] - (1 + 0.25 * Math.Sqrt(2)) * (1 + 0.25 * Math.Sqrt(2)));");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocityX(){
        return new Formula("BoundaryValues.InletVelocityX", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InletVelocityY(){
        return new Formula("BoundaryValues.InletVelocityY", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_PreDisplacementX(){
        return new Formula("BoundaryValues.PreDisplacementX", true, AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_PreDisplacementY(){
        return new Formula("BoundaryValues.PreDisplacementY", true, AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_Pressure(){
        return new Formula("BoundaryValues.Pressure", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [19]:
public static ZLS_Control Couette( int p = 2, int h = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    //int D = 2;
    AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Steady;

    #region db
    C.savetodb = true;
    C.ProjectName = "Couette";
    C.SessionName = "Fluid-Fluid-Taylor-Couette-p" + p + "-h" + h;
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());
    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 0.1;
    C.PhysicalParameters.rho_B = 1.3;
    C.PhysicalParameters.mu_A = 0.01;
    C.PhysicalParameters.mu_B = 0.2;
    double sigma = 0.9;
    C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        //Lame2 = 1000,
        //Viscosity = 0.1
        Lame2 = 1e3,
        Viscosity = 0.2
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid(h));
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#B", BoundaryValueFactory.Get_InletVelocityX());
    C.AddInitialValue("VelocityY#B", BoundaryValueFactory.Get_InletVelocityY());
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_PreDisplacementX());
    C.AddInitialValue("VelocityY#A", BoundaryValueFactory.Get_PreDisplacementY());
    //C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_Zero());
    //C.AddInitialValue("Pressure#C", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi1());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower", "VelocityX#B", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_lower", "VelocityY#B", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_upper", "VelocityX#B", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_upper", "VelocityY#B", BoundaryValueFactory.Get_InletVelocityY());
    //C.AddBoundaryValue("pressure_outlet_upper");
    C.AddBoundaryValue("wall_left", "VelocityX#B", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_left", "VelocityY#B", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_right", "VelocityX#B", BoundaryValueFactory.Get_InletVelocityX());
    C.AddBoundaryValue("wall_right", "VelocityY#B", BoundaryValueFactory.Get_InletVelocityY());
    C.AddBoundaryValue("wall_inlet", "VelocityX#A", BoundaryValueFactory.Get_PreDisplacementX());
    C.AddBoundaryValue("wall_inlet", "VelocityY#A", BoundaryValueFactory.Get_PreDisplacementY());    
    //C.AddBoundaryValue("velocity_inlet_inside", "DisplacementX#C", BoundaryValueFactory.Get_PreDisplacementX());
    //C.AddBoundaryValue("velocity_inlet_inside", "DisplacementY#C", BoundaryValueFactory.Get_PreDisplacementY());
    //C.AddBoundaryValue("wall_inside");



    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    C.PhysicalParameters.sliplength = 0.0;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 2;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    C.AdaptiveMeshRefinement = false;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    C.AMR_startUpSweeps = AMRlvl;
    #endregion

    C.DynamicLoadBalancing_On = false;
    //C.DynamicLoadBalancing_Period = 500;
    C.DynamicLoadBalancing_RedistributeAtStartup = false;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = compMode;
    //double dt = 1;
    //C.dtMax = dt;
    //C.dtMin = dt;
    //C.NoOfTimesteps = 10;
    C.saveperiod = 1;
    #endregion
    return C;
}

## Send and run jobs

In [22]:
double[] Density = new double[] {0.1}; 
int[] Resolution = new int[] {3, 4, 5, 6}; 
int[] DgOrder = new int[] {1, 2, 3}; 

In [24]:
foreach(int DgO in DgOrder){
foreach(int Res in Resolution){
foreach(double Den in Density){
    var C_CTRLFILE = Couette(DgO, Res, 0, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 1;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate(myBatch);
    JobLocal.ShowOutput(); 
}
}
}



Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Fluid-Fluid-Taylor-Couette-p1-h3 ... 
Set Database: { Session Count = 0; Grid Count = 0; Path = D:\Users\miao\Documents\Database\EE-FF-Taylor-Couette-Flow }
Grid is not in database yet...
Grid successfully saved: 8104c140-bc72-4157-b39d-e7d9a099a107


Deploying executables and additional files ...
Deployment directory: D:\Users\miao\Documents\Database\Deployment\EE-FF-Taylor-Couette-Flow-ZwoLevelSetSolver2024Aug21_115444.213073
copied 46 files.
   written file: control.obj
deployment finished.
Starting mini batch processor in external process...
Started mini batch processor on local machine, process id is 258812.
started.
Starting external console ...
(You may close the new window at any time, the job will continue.)


Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


...

Deploying job Fluid-Fluid-Taylor-Couette-p3-h5 ... 
Set Database: { Session Count = 1; Grid Count = 10; Path = D:\Users\miao\Documents\Database\EE-FF-Taylor-Couette-Flow }
Grid is not in database yet...
Grid successfully saved: 582c2159-3090-42a5-af12-fa453652267d


Deploying executables and additional files ...
Deployment directory: D:\Users\miao\Documents\Database\Deployment\EE-FF-Taylor-Couette-Flow-ZwoLevelSetSolver2024Aug21_115521.969874
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.
Starting external console ...
(You may close the new window at any time, the job will continue.)


Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Fluid-Fluid-Taylor-Couette-p3-h6 ... 
Set Database: { Session Count = 3; Grid Count = 11; Path = D:\Users\miao\Documents\Database\EE-FF-Taylor-Couette-Flow }
Grid is not in database yet...
Grid successfully saved: bb7cfd1a-2acb-4d3e-b01b-290a2f1809e6


Deploying executables and additional files ...
Deployment directory: D:\Users\miao\Documents\Database\Deployment\EE-FF-Taylor-Couette-Flow-ZwoLevelSetSolver2024Aug21_115526.698053
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.
Starting external console ...
(You may close the new window at any time, the job will continue.)


In [66]:
wmg.Sessions

#0: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h6	08/21/2024 11:56:01	95101809...
#1: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h6	08/21/2024 11:55:37	d48b5847...
#2: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h4	08/21/2024 11:55:48	8e06e129...
#3: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h5	08/21/2024 11:55:48	cf94bdec...
#4: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h5	08/21/2024 11:55:42	69ac1b0e...
#5: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p1-h6	08/21/2024 11:55:29	36b1bae3...
#6: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h3	08/21/2024 11:55:42	0a36d7fa...
#7: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h4	08/21/2024 11:55:33	454b8825...
#8: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p1-h5	08/21/2024 11:55:23	60d107ac...
#9: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h3	08/21/2024 11:55:32	6953c77c...
#10: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-

In [34]:
wmg.Sessions[0].Timesteps.Count

2

In [64]:
var outPath = wmg.Sessions[1].Timesteps.Every(1).Skip(1).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\EE-FF-Taylor-Couette-Flow__Fluid-Fluid-Taylor-Couette-p2-h6__d48b5847-63db-4998-9ded-3c47ee798f45


## Post processing

In [35]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [37]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [68]:
wmg.Sessions

#0: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h6	08/21/2024 11:56:01	95101809...
#1: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h6	08/21/2024 11:55:37	d48b5847...
#2: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h4	08/21/2024 11:55:48	8e06e129...
#3: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h5	08/21/2024 11:55:48	cf94bdec...
#4: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h5	08/21/2024 11:55:42	69ac1b0e...
#5: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p1-h6	08/21/2024 11:55:29	36b1bae3...
#6: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p3-h3	08/21/2024 11:55:42	0a36d7fa...
#7: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h4	08/21/2024 11:55:33	454b8825...
#8: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p1-h5	08/21/2024 11:55:23	60d107ac...
#9: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-Couette-p2-h3	08/21/2024 11:55:32	6953c77c...
#10: EE-FF-Taylor-Couette-Flow	Fluid-Fluid-Taylor-

In [70]:
var session = wmg.Sessions[0];

In [72]:
session.Timesteps.Count

2

In [78]:
//var mydataset = session.Timesteps.ToDataSet(
//        t => t.PhysicalTime,
//        t => t.Fields.Find("DisplacementX").ProbeAt(0.0, -1.3),
//        t => "DisplacementX"
//        );

In [80]:
//mydataset.PlotNow()

In [ ]:
for (int i = 0; i < 5; i++) 
{
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  //Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

In [ ]:
int[] Cases = new int[] {0, 1, 3, 4}; 

In [ ]:
foreach(int i in Cases){
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

In [50]:
var DispX = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();

In [51]:
DispX.ProbeAt(0.0, 0.25)

0.0008327781162742156

In [53]:
Displacement


(1,1): error CS0103: The name 'Displacement' does not exist in the current context



Error: compilation error

## Restart

In [ ]:
databases[0].Sessions

In [ ]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[0];
Sloc

In [ ]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [ ]:
c2.GetType()

In [ ]:
c2.GridGuid

In [ ]:
Sloc.Timesteps.Last().GridID

In [ ]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [ ]:
c2.GridGuid

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].